In [28]:
import pandas as pd
import numpy as np


In [29]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate a time-series dataset of sales
dates = pd.date_range(start='2023-01-01', periods=100, freq='D')
categories = ['Electronics', 'Clothing', 'Home']

df = pd.DataFrame({
    'date': np.random.choice(dates, size=500),
    'category': np.random.choice(categories, size=500),
    'sales': np.random.uniform(10, 500, size=500).round(2),
    'quantity': np.random.randint(1, 10, size=500),
    'customer_id': np.random.randint(1000, 1050, size=500)
})

# Sort by date
df = df.sort_values('date').reset_index(drop=True)
df.head()

,date,category,sales,quantity,customer_id
0,2023-01-01,Home,61.01,5,1020
1,2023-01-01,Electronics,289.25,2,1015
2,2023-01-01,Clothing,213.17,6,1015
3,2023-01-01,Home,140.84,3,1037
4,2023-01-01,Home,275.40,4,1027


In [30]:
df

,date,category,sales,quantity,customer_id
0,2023-01-01,Home,61.01,5,1020
1,2023-01-01,Electronics,289.25,2,1015
2,2023-01-01,Clothing,213.17,6,1015
3,2023-01-01,Home,140.84,3,1037
4,2023-01-01,Home,275.40,4,1027
...,...,...,...,...,...
495,2023-04-09,Clothing,127.00,4,1020
496,2023-04-09,Clothing,474.17,9,1045
497,2023-04-10,Clothing,59.87,2,1043
498,2023-04-10,Electronics,253.22,8,1031


In [31]:
df['date'] = pd.to_datetime(df['date'])


In [32]:
df = df.sort_values('date')

In [33]:
df

,date,category,sales,quantity,customer_id
0,2023-01-01,Home,61.01,5,1020
1,2023-01-01,Electronics,289.25,2,1015
2,2023-01-01,Clothing,213.17,6,1015
3,2023-01-01,Home,140.84,3,1037
4,2023-01-01,Home,275.40,4,1027
...,...,...,...,...,...
495,2023-04-09,Clothing,127.00,4,1020
496,2023-04-09,Clothing,474.17,9,1045
498,2023-04-10,Electronics,253.22,8,1031
497,2023-04-10,Clothing,59.87,2,1043


In [34]:
df2 = df.set_index('date')

In [35]:
df2

,category,sales,quantity,customer_id
date,,,,
2023-01-01,Home,61.01,5,1020
2023-01-01,Electronics,289.25,2,1015
2023-01-01,Clothing,213.17,6,1015
2023-01-01,Home,140.84,3,1037
2023-01-01,Home,275.40,4,1027
...,...,...,...,...
2023-04-09,Clothing,127.00,4,1020
2023-04-09,Clothing,474.17,9,1045
2023-04-10,Electronics,253.22,8,1031


In [36]:
df2['rolling_7_sales'] = df2['sales'].rolling('7d').sum()

In [37]:
df2

,category,sales,quantity,customer_id,rolling_7_sales
date,,,,,
2023-01-01,Home,61.01,5,1020,61.01
2023-01-01,Electronics,289.25,2,1015,350.26
2023-01-01,Clothing,213.17,6,1015,563.43
2023-01-01,Home,140.84,3,1037,704.27
2023-01-01,Home,275.40,4,1027,979.67
...,...,...,...,...,...
2023-04-09,Clothing,127.00,4,1020,10497.95
2023-04-09,Clothing,474.17,9,1045,10972.12
2023-04-10,Electronics,253.22,8,1031,9106.68


In [38]:
df.to_csv('data.csv')

**Question 2: Cohort Analysis (Datetime Manipulation)**
Find the exact date each `customer_id` made their _first_ purchase. Then, calculate how many _unique_ customers made their first purchase in each month.

In [39]:
df['date'] = pd.to_datetime(df['date'])

In [40]:
new_customers =(
    df.groupby('customer_id')['date']
    .min()
    .dt.to_period('M')
    .value_counts()
    .sort_index()
)

print(new_customers)

date
2023-01    48
2023-02     2
Freq: M, Name: count, dtype: int64


cohort_month
2023-01    48
2023-02     2
Freq: M, Name: customer_id, dtype: int64


**Question 3: Custom Groupby Aggregation (Top N Items)**
For each `category`, find the top 2 days that had the highest total sales.

In [46]:
daily_cat_sales = df.groupby(['category','date'])['sales'].sum().reset_index()

def get_top_2(group):
    return group.nlargest(2,'sales')


top_days = daily_cat_sales.groupby('category').apply(get_top_2).reset_index(drop=True)

top_days

C:\Users\francis\AppData\Local\Temp\ipykernel_2248\1536711673.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_days = daily_cat_sales.groupby('category').apply(get_top_2).reset_index(drop=True)


,category,date,sales
0,Clothing,2023-02-17,1777.93
1,Clothing,2023-01-08,1605.99
2,Electronics,2023-01-02,1804.24
3,Electronics,2023-03-21,1495.55
4,Home,2023-02-21,1362.62
5,Home,2023-04-09,1335.31


In [48]:
# 1. Extract the month number from the date
df['month'] = df['date'].dt.month

# 2. Create the pivot table
pivot_df = pd.pivot_table(
    df,
    values='quantity',
    index='category',   # Rows
    columns='month',    # Columns
    aggfunc='sum',      # Strategy
    fill_value=0        # Replace NaN with 0
)

print(pivot_df)

month          1    2    3   4
category                      
Clothing     233  249  198  95
Electronics  291  238  270  75
Home         217  249  269  78
